Project Idea -> Consume SpaceX Rockets Api and use Pydantic to build a data models with validation

Utilization data in api json
    -description -> OK
    -cost_per_launch -> OK
    -company -> OK
    -success_rate_pct -> OK
    -height -> OK
    -country -> OK
    -first_flight -> OK
    -rocket_name -> OK

In [155]:
#imports
import requests
from pydantic import BaseModel, Field,model_validator,conint, computed_field
from pydantic_settings import BaseSettings,SettingsConfigDict
from dotenv import load_dotenv
import os
from typing import List
from datetime import date

In [156]:
#request
load_dotenv(dotenv_path="env_practice")
response_data = requests.get(os.environ["API_SPACEX_ROCKET"])

In [172]:
#class model
class Dimension(BaseModel):
    meters: float
    feet: float
    
    @model_validator(mode="before")
    @classmethod
    def validate_type(cls, values):
        if not isinstance(values['meters'], float):
            values['meters'] = float(values['meters'])
        if not isinstance(values['feet'], float):
            values['feet'] = float(values['feet'])    
        return values

class PayloadWeights(BaseModel):
    id: str
    name: str
    kg: int
    lb: int
        
    
class Details(BaseModel):
    country: str
    first_flight: date
    cost_per_launch: float
    rocket_name: str
    success_rate_pct: conint(ge=0, le=100)
    description: str
    
    @model_validator(mode="before")
    def validate_type(cls, values):
        if not isinstance(values['cost_per_launch'], float):
            values['cost_per_launch'] = float(values['cost_per_launch'])
        if not isinstance(values['success_rate_pct'], float):
            values['success_rate_pct'] = float(values['success_rate_pct'])
        return values
    
    @computed_field
    @property
    def rocket_age(self) -> int:
        rocket_age = date.today() - self.first_flight
        return rocket_age.days

class Data(BaseModel):
    company: str = Field(alias="company")
    height: Dimension
    diameter: Dimension
    payload_weights: List[PayloadWeights]
    details: Details = None
    
    @computed_field
    @property
    def total_kg(self) -> int:
        return sum(r.kg for r in self.payload_weights)

In [174]:
data = Data(**response_data.json())

details_init = Details(country=response_data.json().get('country'), cost_per_launch=response_data.json().get('cost_per_launch'),success_rate_pct=response_data.json().get('success_rate_pct'),first_flight=response_data.json().get('first_flight'), rocket_name=response_data.json().get('rocket_name'), description=response_data.json().get('description'))

data.details = details_init

print(data.model_dump_json())

{"company":"SpaceX","height":{"meters":70.0,"feet":229.6},"diameter":{"meters":3.7,"feet":12.0},"payload_weights":[{"id":"leo","name":"Low Earth Orbit","kg":22800,"lb":50265},{"id":"gto","name":"Geosynchronous Transfer Orbit","kg":8300,"lb":18300},{"id":"mars","name":"Mars Orbit","kg":4020,"lb":8860}],"details":{"country":"United States","first_flight":"2010-06-04","cost_per_launch":50000000.0,"rocket_name":"Falcon 9","success_rate_pct":97,"description":"Falcon 9 is a two-stage rocket designed and manufactured by SpaceX for the reliable and safe transport of satellites and the Dragon spacecraft into orbit.","rocket_age":5766},"total_kg":35120}
